# Notebook 06 — AIA AlexNet Fold-2015 Full Benchmark

This notebook is generated from the repaired Notebook 05 pilot benchmark, but is configured for `RUN_MODE = "full"`. It should be run only when the GPU VM is intentionally active. After completion, back up outputs to GCS/GitHub and stop the VM immediately.

Main evaluation rule: select the threshold on validation by maximum TSS, then apply that threshold unchanged to the 2015 test split.


# 05 — AIA AlexNet-Style CNN Benchmark on Formal Fold `test_2015` — FIXED

This notebook trains the first formal AIA image CNN benchmark after Notebook 04.

Protocol:

```text
fold_id: test_2015
train: 2010–2013
validation: 2014
test: 2015
```

It reports ROC-AUC, PR-AUC, Brier score, TSS, HSS, accuracy, precision, recall, specificity, F1, and confusion matrix.

Important rule: select threshold on validation by maximum TSS, then apply that threshold unchanged to test.

In [ ]:
from pathlib import Path
import json, time, random, hashlib, subprocess, warnings
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, confusion_matrix

warnings.filterwarnings("ignore")

ROOT = Path.home() / "solar_flare_aia"
METRICS_DIR = ROOT / "results/metrics"
MODELS_DIR = ROOT / "results/models"
CACHE_DIR = ROOT / "cache/gcs_npz_alexnet_fold2015"

for p in [METRICS_DIR, MODELS_DIR, CACHE_DIR]:
    p.mkdir(parents=True, exist_ok=True)

FOLD_ASSIGNMENTS_RAW = METRICS_DIR / "aia_baseline_2010_2016_protocol_fold_assignments.csv"
FOLD_ASSIGNMENTS_GZ = METRICS_DIR / "aia_baseline_2010_2016_protocol_fold_assignments.csv.gz"

FOLD_ID = "test_2015"
EXPERIMENT_NAME = "aia_alexnet_fold2015_largecap_benchmark"

RUN_MODE = "largecap"  # large-capped fold-2015 benchmark

# Large-capped setting: keep all positives, cap negatives for speed/cost control.
LARGECAP_TRAIN_POS = None
LARGECAP_TRAIN_NEG = 10000
LARGECAP_VAL_POS = None
LARGECAP_VAL_NEG = 5000
LARGECAP_TEST_POS = None
LARGECAP_TEST_NEG = 5000

PILOT_TRAIN_POS = 500
PILOT_TRAIN_NEG = 5000
PILOT_VAL_POS = 300
PILOT_VAL_NEG = 3000
PILOT_TEST_POS = 300
PILOT_TEST_NEG = 3000

IMAGE_SIZE = 224
BATCH_SIZE = 16
NUM_EPOCHS = 6
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 2

SEED = 77
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Root:", ROOT)
print("Run mode:", RUN_MODE)
print("Fold:", FOLD_ID)
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 1. Load fold assignments from Notebook 04

In [ ]:
if FOLD_ASSIGNMENTS_GZ.exists():
    fold_path = FOLD_ASSIGNMENTS_GZ
elif FOLD_ASSIGNMENTS_RAW.exists():
    fold_path = FOLD_ASSIGNMENTS_RAW
else:
    msg = (
        "Could not find fold assignment file. Expected either:\n"
        + str(FOLD_ASSIGNMENTS_GZ)
        + "\nor\n"
        + str(FOLD_ASSIGNMENTS_RAW)
        + "\nRun Notebook 04 first."
    )
    raise FileNotFoundError(msg)

print("Loading:", fold_path)
assignments = pd.read_csv(fold_path, low_memory=False)

fold_df = assignments[assignments["fold_id"] == FOLD_ID].copy()
if len(fold_df) == 0:
    raise ValueError("No rows found for fold_id=" + str(FOLD_ID))

fold_df["label_48h_final"] = fold_df["label_48h_final"].astype(int)
fold_df["year"] = fold_df["year"].astype(int)

print("Rows in selected fold:", len(fold_df))
display(
    fold_df.groupby(["split", "year", "label_48h_final"])
    .size()
    .rename("count")
    .reset_index()
)

## 2. Build train/validation/test dataframes

In [ ]:
def stratified_cap(split_df, pos_cap=None, neg_cap=None, seed=SEED):
    pos = split_df[split_df["label_48h_final"] == 1]
    neg = split_df[split_df["label_48h_final"] == 0]
    if pos_cap is not None and len(pos) > pos_cap:
        pos = pos.sample(n=pos_cap, random_state=seed)
    if neg_cap is not None and len(neg) > neg_cap:
        neg = neg.sample(n=neg_cap, random_state=seed)
    out = pd.concat([pos, neg], ignore_index=True)
    return out.sample(frac=1.0, random_state=seed).reset_index(drop=True)

train_full = fold_df[fold_df["split"] == "train"].copy()
val_full = fold_df[fold_df["split"] == "val"].copy()
test_full = fold_df[fold_df["split"] == "test"].copy()

if RUN_MODE == "pilot":
    train_df = stratified_cap(train_full, PILOT_TRAIN_POS, PILOT_TRAIN_NEG)
    val_df = stratified_cap(val_full, PILOT_VAL_POS, PILOT_VAL_NEG)
    test_df = stratified_cap(test_full, PILOT_TEST_POS, PILOT_TEST_NEG)
elif RUN_MODE == "largecap":
    train_df = stratified_cap(train_full, LARGECAP_TRAIN_POS, LARGECAP_TRAIN_NEG)
    val_df = stratified_cap(val_full, LARGECAP_VAL_POS, LARGECAP_VAL_NEG)
    test_df = stratified_cap(test_full, LARGECAP_TEST_POS, LARGECAP_TEST_NEG)
elif RUN_MODE == "full":
    train_df = train_full.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    val_df = val_full.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    test_df = test_full.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
else:
    raise ValueError("RUN_MODE must be pilot, largecap, or full")

def split_summary(name, d):
    return {
        "split": name,
        "rows": int(len(d)),
        "positives": int(d["label_48h_final"].sum()),
        "negatives": int((d["label_48h_final"] == 0).sum()),
        "positive_rate": float(d["label_48h_final"].mean()),
        "years": str(sorted(d["year"].unique().tolist())),
    }

summary_df = pd.DataFrame([split_summary("train", train_df), split_summary("val", val_df), split_summary("test", test_df)])
display(summary_df)

train_df.to_csv(METRICS_DIR / f"{EXPERIMENT_NAME}_{RUN_MODE}_train_samples.csv", index=False)
val_df.to_csv(METRICS_DIR / f"{EXPERIMENT_NAME}_{RUN_MODE}_val_samples.csv", index=False)
test_df.to_csv(METRICS_DIR / f"{EXPERIMENT_NAME}_{RUN_MODE}_test_samples.csv", index=False)

## 3. Dataset and GCS cache helpers

In [ ]:
def local_cache_path_from_gcs(gcs_path):
    h = hashlib.md5(str(gcs_path).encode("utf-8")).hexdigest()
    return CACHE_DIR / (h + "_" + Path(str(gcs_path)).name)

def ensure_local_npz(gcs_path):
    local_path = local_cache_path_from_gcs(gcs_path)
    if local_path.exists() and local_path.stat().st_size > 0:
        return local_path
    cmd = ["gcloud", "storage", "cp", str(gcs_path), str(local_path)]
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    if result.returncode != 0:
        if local_path.exists():
            local_path.unlink()
        msg = "Failed to copy " + str(gcs_path) + "\nSTDOUT:\n" + result.stdout + "\nSTDERR:\n" + result.stderr
        raise RuntimeError(msg)
    return local_path

class AIAFoldDataset(Dataset):
    def __init__(self, frame, image_size=224):
        self.frame = frame.reset_index(drop=True)
        self.image_size = image_size

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        local_path = ensure_local_npz(row["gcp_path"])
        with np.load(local_path, allow_pickle=True) as data:
            x = data["x"].astype(np.float32)
        if x.ndim != 3 or x.shape[-1] != 6:
            raise ValueError("Unexpected x shape for " + str(row["sample_id"]) + ": " + str(x.shape))
        x = torch.from_numpy(x).permute(2, 0, 1)
        if self.image_size is not None and x.shape[-1] != self.image_size:
            x = F.interpolate(x.unsqueeze(0), size=(self.image_size, self.image_size), mode="bilinear", align_corners=False).squeeze(0)
        y = torch.tensor(np.float32(row["label_48h_final"]), dtype=torch.float32)
        return {"x": x, "y": y, "sample_id": row["sample_id"], "year": int(row["year"])}

train_ds = AIAFoldDataset(train_df, IMAGE_SIZE)
val_ds = AIAFoldDataset(val_df, IMAGE_SIZE)
test_ds = AIAFoldDataset(test_df, IMAGE_SIZE)
print("Dataset sizes:", len(train_ds), len(val_ds), len(test_ds))

## 4. Pre-cache files

In [ ]:
def precache_dataset(ds, name):
    start = time.time()
    failures = []
    for i in range(len(ds.frame)):
        gcs_path = ds.frame.iloc[i]["gcp_path"]
        sample_id = ds.frame.iloc[i]["sample_id"]
        try:
            ensure_local_npz(gcs_path)
        except Exception as e:
            failures.append({"split": name, "idx": i, "sample_id": sample_id, "gcp_path": gcs_path, "error": str(e)})
        if (i + 1) % 500 == 0 or (i + 1) == len(ds.frame):
            print(f"{name}: cached/checked {i+1}/{len(ds.frame)}")
    print(f"{name}: done in {(time.time() - start)/60:.2f} minutes. failures={len(failures)}")
    return failures

all_failures = []
all_failures += precache_dataset(train_ds, "train")
all_failures += precache_dataset(val_ds, "val")
all_failures += precache_dataset(test_ds, "test")

if all_failures:
    fail_df = pd.DataFrame(all_failures)
    fail_path = METRICS_DIR / f"{EXPERIMENT_NAME}_{RUN_MODE}_cache_failures.csv"
    fail_df.to_csv(fail_path, index=False)
    display(fail_df.head())
    raise RuntimeError("Cache failures detected. Saved to " + str(fail_path))
print("All requested files cached.")

## 5. Data loaders

In [ ]:
train_labels = train_df["label_48h_final"].values.astype(int)
class_counts = np.bincount(train_labels, minlength=2)
class_weights = 1.0 / np.maximum(class_counts, 1)
sample_weights = class_weights[train_labels]

sampler = WeightedRandomSampler(weights=torch.DoubleTensor(sample_weights), num_samples=len(train_labels), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())

batch = next(iter(train_loader))
print("x batch:", tuple(batch["x"].shape))
print("y batch:", tuple(batch["y"].shape))
print("positives in batch:", int(batch["y"].sum().item()))
print("x min/max:", float(batch["x"].min()), float(batch["x"].max()))

## 6. Model

In [ ]:
class AlexNetAIA(nn.Module):
    def __init__(self, in_channels=6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(in_channels, 64, kernel_size=11, stride=4, padding=2), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(64, 192, kernel_size=5, padding=2), nn.BatchNorm2d(192), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=3, stride=2),
            nn.Conv2d(192, 384, kernel_size=3, padding=1), nn.BatchNorm2d(384), nn.ReLU(inplace=True),
            nn.Conv2d(384, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(kernel_size=3, stride=2),
        )
        self.pool = nn.AdaptiveAvgPool2d((6, 6))
        self.classifier = nn.Sequential(
            nn.Dropout(0.5), nn.Linear(256 * 6 * 6, 512), nn.ReLU(inplace=True),
            nn.Dropout(0.5), nn.Linear(512, 128), nn.ReLU(inplace=True), nn.Linear(128, 1),
        )
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = torch.flatten(x, 1)
        return self.classifier(x).squeeze(1)

model = AlexNetAIA(in_channels=6).to(DEVICE)
n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("Model:", model.__class__.__name__)
print("Trainable parameters:", n_params)
with torch.no_grad():
    logits = model(batch["x"].to(DEVICE))
print("Test logits:", tuple(logits.shape))

## 7. Metrics

In [ ]:
def safe_auc(fn, y_true, y_score):
    y_true = np.asarray(y_true)
    if len(np.unique(y_true)) < 2:
        return float("nan")
    return float(fn(y_true, y_score))

def compute_threshold_metrics(y_true, y_prob, threshold):
    y_true = np.asarray(y_true).astype(int)
    y_prob = np.asarray(y_prob)
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    accuracy = (tp + tn) / max(tp + tn + fp + fn, 1)
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    specificity = tn / max(tn + fp, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    tss = recall + specificity - 1
    denom = ((tp + fn) * (fn + tn) + (tp + fp) * (fp + tn))
    hss = 2 * (tp * tn - fp * fn) / denom if denom != 0 else float("nan")
    return {"threshold": float(threshold), "accuracy": float(accuracy), "precision": float(precision), "recall": float(recall), "specificity": float(specificity), "f1": float(f1), "tss": float(tss), "hss": float(hss), "tp": int(tp), "tn": int(tn), "fp": int(fp), "fn": int(fn)}

def find_best_tss_threshold(y_true, y_prob):
    thresholds = np.linspace(0.0, 1.0, 401)
    rows = [compute_threshold_metrics(y_true, y_prob, t) for t in thresholds]
    best = max(rows, key=lambda r: (r["tss"], r["hss"], r["f1"]))
    return best, pd.DataFrame(rows)

def summarize_predictions(y_true, y_prob, selected_threshold=None):
    out = {
        "roc_auc": safe_auc(roc_auc_score, y_true, y_prob),
        "pr_auc": safe_auc(average_precision_score, y_true, y_prob),
        "brier_score": float(brier_score_loss(y_true, y_prob)),
        "at_0_5": compute_threshold_metrics(y_true, y_prob, 0.5),
    }
    best_tss, threshold_grid = find_best_tss_threshold(y_true, y_prob)
    out["best_tss"] = best_tss
    if selected_threshold is not None:
        out["at_selected_threshold"] = compute_threshold_metrics(y_true, y_prob, selected_threshold)
    return out, threshold_grid

## 8. Training functions

In [ ]:
def run_epoch(model, loader, optimizer=None, criterion=None):
    is_train = optimizer is not None
    model.train(is_train)
    losses, all_y, all_prob, all_sample_id = [], [], [], []
    for batch in loader:
        x = batch["x"].to(DEVICE, non_blocking=True)
        y = batch["y"].to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(is_train):
            logits = model(x)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()
        prob = torch.sigmoid(logits).detach().cpu().numpy()
        losses.append(float(loss.detach().cpu().item()))
        all_y.extend(y.detach().cpu().numpy().astype(int).tolist())
        all_prob.extend(prob.tolist())
        all_sample_id.extend(batch["sample_id"])
    return {"loss": float(np.mean(losses)), "y_true": np.array(all_y, dtype=int), "y_prob": np.array(all_prob, dtype=float), "sample_id": all_sample_id}

train_pos = int(train_df["label_48h_final"].sum())
train_neg = int((train_df["label_48h_final"] == 0).sum())
pos_weight_value = train_neg / max(train_pos, 1)

criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_value], device=DEVICE))
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print("Train positives:", train_pos)
print("Train negatives:", train_neg)
print("pos_weight:", pos_weight_value)

## 9. Train and select threshold on validation

In [ ]:
history = []
best_epoch = None
best_val_tss = -999.0
MODEL_PATH = MODELS_DIR / f"{EXPERIMENT_NAME}_{RUN_MODE}.pt"

start_all = time.time()

for epoch in range(1, NUM_EPOCHS + 1):
    start = time.time()
    train_out = run_epoch(model, train_loader, optimizer=optimizer, criterion=criterion)
    val_out = run_epoch(model, val_loader, optimizer=None, criterion=criterion)

    train_metrics, _ = summarize_predictions(train_out["y_true"], train_out["y_prob"])
    val_metrics, _ = summarize_predictions(val_out["y_true"], val_out["y_prob"])
    val_best = val_metrics["best_tss"]

    row = {
        "epoch": int(epoch),
        "train_loss": train_out["loss"],
        "val_loss": val_out["loss"],
        "train_roc_auc": train_metrics["roc_auc"],
        "train_pr_auc": train_metrics["pr_auc"],
        "val_roc_auc": val_metrics["roc_auc"],
        "val_pr_auc": val_metrics["pr_auc"],
        "val_brier_score": val_metrics["brier_score"],
        "val_tss_at_0_5": val_metrics["at_0_5"]["tss"],
        "val_f1_at_0_5": val_metrics["at_0_5"]["f1"],
        "best_tss_threshold": val_best["threshold"],
        "val_best_tss": val_best["tss"],
        "val_best_hss": val_best["hss"],
        "val_best_precision": val_best["precision"],
        "val_best_recall": val_best["recall"],
        "val_best_specificity": val_best["specificity"],
        "val_best_tp": val_best["tp"],
        "val_best_tn": val_best["tn"],
        "val_best_fp": val_best["fp"],
        "val_best_fn": val_best["fn"],
        "elapsed_min": (time.time() - start) / 60,
    }
    history.append(row)

    print("")
    print("Epoch", epoch, "/", NUM_EPOCHS)
    print("Train loss:", row["train_loss"], "Val loss:", row["val_loss"])
    print("Val ROC-AUC:", row["val_roc_auc"], "Val PR-AUC:", row["val_pr_auc"])
    print("Val @0.5:", val_metrics["at_0_5"])
    print("Val @best TSS:", val_best)
    print("Elapsed min:", round(row["elapsed_min"], 2))

    if val_best["tss"] > best_val_tss:
        best_val_tss = val_best["tss"]
        best_epoch = int(epoch)
        torch.save({
            "model_state_dict": model.state_dict(),
            "epoch": int(epoch),
            "val_best_tss": float(best_val_tss),
            "val_best_threshold": float(val_best["threshold"]),
            "config": {"experiment_name": EXPERIMENT_NAME, "run_mode": RUN_MODE, "fold_id": FOLD_ID, "image_size": IMAGE_SIZE, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "weight_decay": WEIGHT_DECAY, "pos_weight": float(pos_weight_value), "seed": SEED},
        }, MODEL_PATH)

history_df = pd.DataFrame(history)
display(history_df)
print("Best epoch:", best_epoch)
print("Best validation TSS:", best_val_tss)
print("Saved model:", MODEL_PATH)
print("Total elapsed min:", round((time.time() - start_all) / 60, 2))

## 10. Evaluate best checkpoint on validation and test

In [ ]:
checkpoint = torch.load(MODEL_PATH, map_location=DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)
selected_threshold = float(checkpoint["val_best_threshold"])

val_out = run_epoch(model, val_loader, optimizer=None, criterion=criterion)
test_out = run_epoch(model, test_loader, optimizer=None, criterion=criterion)

val_metrics, val_threshold_grid = summarize_predictions(val_out["y_true"], val_out["y_prob"])
test_metrics, test_threshold_grid = summarize_predictions(test_out["y_true"], test_out["y_prob"], selected_threshold=selected_threshold)

print("Selected threshold from validation:", selected_threshold)

val_table = pd.DataFrame([{"split": "val", "roc_auc": val_metrics["roc_auc"], "pr_auc": val_metrics["pr_auc"], "brier_score": val_metrics["brier_score"], **val_metrics["best_tss"]}])
test_sel = test_metrics["at_selected_threshold"]
test_table = pd.DataFrame([{"split": "test", "roc_auc": test_metrics["roc_auc"], "pr_auc": test_metrics["pr_auc"], "brier_score": test_metrics["brier_score"], **test_sel}])

display(Markdown("### Validation metrics"))
display(val_table)
display(Markdown("### Test metrics at validation-selected threshold"))
display(test_table)

## 11. Save outputs

In [ ]:
METRICS_JSON = METRICS_DIR / f"{EXPERIMENT_NAME}_{RUN_MODE}_metrics.json"
HISTORY_CSV = METRICS_DIR / f"{EXPERIMENT_NAME}_{RUN_MODE}_history.csv"
VAL_PRED_CSV = METRICS_DIR / f"{EXPERIMENT_NAME}_{RUN_MODE}_val_predictions.csv"
TEST_PRED_CSV = METRICS_DIR / f"{EXPERIMENT_NAME}_{RUN_MODE}_test_predictions.csv"
VAL_GRID_CSV = METRICS_DIR / f"{EXPERIMENT_NAME}_{RUN_MODE}_val_threshold_grid.csv"
TEST_GRID_CSV = METRICS_DIR / f"{EXPERIMENT_NAME}_{RUN_MODE}_test_threshold_grid.csv"
SUMMARY_MD = METRICS_DIR / f"{EXPERIMENT_NAME}_{RUN_MODE}_readable_summary.md"

history_df.to_csv(HISTORY_CSV, index=False)
val_threshold_grid.to_csv(VAL_GRID_CSV, index=False)
test_threshold_grid.to_csv(TEST_GRID_CSV, index=False)

val_pred_df = pd.DataFrame({"sample_id": val_out["sample_id"], "y_true": val_out["y_true"], "y_prob": val_out["y_prob"], "selected_threshold": selected_threshold, "y_pred_selected_threshold": (val_out["y_prob"] >= selected_threshold).astype(int)})
test_pred_df = pd.DataFrame({"sample_id": test_out["sample_id"], "y_true": test_out["y_true"], "y_prob": test_out["y_prob"], "selected_threshold": selected_threshold, "y_pred_selected_threshold": (test_out["y_prob"] >= selected_threshold).astype(int)})
val_pred_df.to_csv(VAL_PRED_CSV, index=False)
test_pred_df.to_csv(TEST_PRED_CSV, index=False)

metrics_payload = {
    "experiment_name": EXPERIMENT_NAME,
    "run_mode": RUN_MODE,
    "fold_id": FOLD_ID,
    "data_summary": summary_df.to_dict(orient="records"),
    "model": {"architecture": "AlexNet-style six-channel CNN", "trainable_parameters": int(n_params), "image_size": int(IMAGE_SIZE), "batch_size": int(BATCH_SIZE), "epochs": int(NUM_EPOCHS), "learning_rate": float(LEARNING_RATE), "weight_decay": float(WEIGHT_DECAY), "pos_weight": float(pos_weight_value), "weighted_random_sampler": True},
    "best_epoch": int(best_epoch),
    "selected_threshold_from_validation": float(selected_threshold),
    "validation": val_metrics,
    "test": test_metrics,
}
METRICS_JSON.write_text(json.dumps(metrics_payload, indent=2))

test_sel = test_metrics["at_selected_threshold"]
summary = f"""# AIA AlexNet Fold 2015 Benchmark Summary

## Run

- Experiment: `{EXPERIMENT_NAME}`
- Mode: `{RUN_MODE}`
- Fold: `{FOLD_ID}`
- Train years: `{sorted(train_df["year"].unique().tolist())}`
- Validation year: `{sorted(val_df["year"].unique().tolist())}`
- Test year: `{sorted(test_df["year"].unique().tolist())}`
- Label: `label_48h_final`
- Threshold rule: validation-selected max TSS, applied unchanged to test

## Data

{summary_df.to_markdown(index=False)}

## Best validation checkpoint

- Best epoch: `{best_epoch}`
- Validation-selected threshold: `{selected_threshold:.4f}`
- Validation ROC-AUC: `{val_metrics["roc_auc"]:.4f}`
- Validation PR-AUC: `{val_metrics["pr_auc"]:.4f}`
- Validation best TSS: `{val_metrics["best_tss"]["tss"]:.4f}`
- Validation best HSS: `{val_metrics["best_tss"]["hss"]:.4f}`

## Test result at validation-selected threshold

- Test ROC-AUC: `{test_metrics["roc_auc"]:.4f}`
- Test PR-AUC: `{test_metrics["pr_auc"]:.4f}`
- Test Brier score: `{test_metrics["brier_score"]:.4f}`
- Test threshold: `{test_sel["threshold"]:.4f}`
- Test Accuracy: `{test_sel["accuracy"]:.4f}`
- Test Precision: `{test_sel["precision"]:.4f}`
- Test Recall: `{test_sel["recall"]:.4f}`
- Test Specificity: `{test_sel["specificity"]:.4f}`
- Test F1: `{test_sel["f1"]:.4f}`
- Test TSS: `{test_sel["tss"]:.4f}`
- Test HSS: `{test_sel["hss"]:.4f}`
- Confusion matrix: TP=`{test_sel["tp"]}`, TN=`{test_sel["tn"]}`, FP=`{test_sel["fp"]}`, FN=`{test_sel["fn"]}`

## Interpretation note

This notebook is configured with `RUN_MODE = "largecap"`, so it keeps the formal fold-2015 chronology but uses all available positives and capped negatives for a cost-controlled benchmark. The validation-selected max-TSS threshold is applied unchanged to the test split for the main reported result.
"""
SUMMARY_MD.write_text(summary)

print("Saved:", METRICS_JSON)
print("Saved:", HISTORY_CSV)
print("Saved:", VAL_PRED_CSV)
print("Saved:", TEST_PRED_CSV)
print("Saved:", SUMMARY_MD)
display(Markdown(summary))

## 12. Backup, commit, and stop VM

After the notebook finishes, run these commands in the VS Code terminal:

```bash
cd ~/solar_flare_aia

gcloud storage cp notebooks/training/05_aia_alexnet_fold2015_benchmark_FIXED.ipynb \
  gs://suryabench-sharp-pipeline-bamidele/training_docs/

gcloud storage cp results/metrics/aia_alexnet_fold2015_benchmark_* \
  gs://suryabench-sharp-pipeline-bamidele/training_docs/

gcloud storage cp results/models/aia_alexnet_fold2015_benchmark_*.pt \
  gs://suryabench-sharp-pipeline-bamidele/training_models/

git add notebooks/training/05_aia_alexnet_fold2015_benchmark_FIXED.ipynb
git add results/metrics/aia_alexnet_fold2015_benchmark_*

git commit -m "Add AlexNet-style AIA fold-2015 benchmark"
git push

gcloud compute instances stop solar-flare-aia-training-l4-c \
  --project=sonorous-shore-450510-i4 \
  --zone=europe-west4-c
```